# 4-Parameter Logistic (4PL) Curve Fit from Excel Data

This notebook reads dose-response data from an Excel file, fits a 4PL curve, and calculates the **IC50**.

### How to use your own data
Replace `sample_data.xlsx` with your own Excel file. Your file must have two columns:
- `Concentration_uM` — compound concentration in µM
- `Inhibition_pct` — measured % inhibition at each concentration

The sheet must be named `Data`.

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

## Step 1: Load data from Excel

In [ ]:
# Load the Excel file
df = pd.read_excel('sample_data.xlsx', sheet_name='Data')

# Extract columns
concentrations = df['Concentration_uM'].values
inhibition     = df['Inhibition_pct'].values

print(df.to_string(index=False))

## Step 2: Define the 4PL equation

$$f(x) = A + \frac{D - A}{1 + \left(\frac{C}{x}\right)^B}$$

- **A** = bottom asymptote (minimum response)
- **B** = Hill slope (steepness of the curve)
- **C** = IC50 (concentration at 50% effect)
- **D** = top asymptote (maximum response)

In [ ]:
def four_param_logistic(x, A, B, C, D):
    """4PL equation — ascending form for % inhibition assays."""
    return A + (D - A) / (1.0 + (C / x) ** B)

## Step 3: Fit the curve

In [ ]:
# Initial guesses: [bottom, hill_slope, ic50_guess, top]
initial_guesses = [
    min(inhibition),          # A: bottom
    1.0,                      # B: Hill slope
    np.median(concentrations),# C: IC50 starting guess
    max(inhibition)           # D: top
]

# Bounds: keep slope and IC50 positive; top/bottom unconstrained
param_bounds = (
    [-np.inf, 0.1, 1e-6, -np.inf],
    [ np.inf, 5.0, 1e6,   np.inf]
)

popt, pcov = curve_fit(
    four_param_logistic,
    concentrations,
    inhibition,
    p0=initial_guesses,
    bounds=param_bounds,
    maxfev=10000
)

A_fit, B_fit, C_fit, D_fit = popt

## Step 4: Results

In [ ]:
# Calculate R²
y_pred = four_param_logistic(concentrations, *popt)
ss_res = np.sum((inhibition - y_pred) ** 2)
ss_tot = np.sum((inhibition - np.mean(inhibition)) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"Bottom asymptote (A): {A_fit:.2f}%")
print(f"Hill slope       (B): {B_fit:.3f}")
print(f"IC50             (C): {C_fit:.4f} µM")
print(f"Top asymptote    (D): {D_fit:.2f}%")
print(f"R²                  : {r2:.5f}")

## Step 5: Plot

In [ ]:
# Smooth curve for plotting
x_smooth = np.logspace(
    np.log10(min(concentrations) / 3),
    np.log10(max(concentrations) * 3),
    300
)
y_smooth = four_param_logistic(x_smooth, *popt)
y_ic50   = four_param_logistic(C_fit, *popt)

fig, ax = plt.subplots(figsize=(8, 6))

# Raw data
ax.scatter(concentrations, inhibition, color='#e74c3c', s=80, zorder=5, label='Raw Data')

# Fitted curve
ax.plot(x_smooth, y_smooth, color='#2980b9', linewidth=2.5, label=f'4PL Fit (IC50 = {C_fit:.3f} µM)')

# IC50 annotation lines (drawn in data coordinates)
ax.plot([C_fit, C_fit], [-5, y_ic50], color='#27ae60', linestyle='--', linewidth=1.5)
ax.plot([min(concentrations) / 3, C_fit], [y_ic50, y_ic50], color='#27ae60', linestyle='--', linewidth=1.5)
ax.text(C_fit * 1.3, y_ic50 + 3, f'IC50 = {C_fit:.3f} µM', color='#27ae60', fontsize=11)

ax.set_xscale('log')
ax.set_xlabel('Concentration (µM)', fontsize=13)
ax.set_ylabel('% Inhibition', fontsize=13)
ax.set_title('Dose-Response 4PL Curve Fit', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, which='both', linestyle='--', alpha=0.4)
ax.set_ylim(-5, 110)

plt.tight_layout()
plt.savefig('4PL_curvefit_plot.png', dpi=150)
plt.show()
print('Plot saved as 4PL_curvefit_plot.png')